In [0]:
# =============================================================================
# CONFIGURAÇÕES E UTILITÁRIOS DE GRAVAÇÃO
# =============================================================================

import json
import re
import pandas as pd
import pyarrow as pa
from datetime import datetime, timezone
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

# Constantes locais
CONTAINER = "squad1"

def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Usa a URI nativa abfss:// com tratamento para gravação na raiz (camada vazia)."""
    conta = storage_opts.get("account_name", "internshipdatalake")

    # Monta só o corpo do caminho (sem o protocolo), evitando barras duplicadas
    corpo = tabela if not camada else f"{camada}/{tabela}"
    corpo = corpo.strip("/")
    corpo = re.sub(r"/+", "/", corpo)  # colapsa // apenas dentro do corpo, nunca no protocolo

    return f"abfss://{CONTAINER}@{conta}.dfs.core.windows.net/{corpo}"

def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    try:
        # Traduz dinamicamente os parâmetros para o formato esperado pelo delta-rs SDK
        opts_sdk = {
            "account_name": storage_opts.get("account_name", "internshipdatalake"),
            "client_id": storage_opts.get("client_id"),
            "client_secret": storage_opts.get("client_secret"),
            "tenant_id": storage_opts.get("tenant_id")
        }
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=opts_sdk)
        return True
    except Exception:
        return False

def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    path = get_delta_path(camada, tabela, storage_opts)
    modo_real = mode if delta_existe(camada, tabela, storage_opts) else "overwrite"

    try:
        # 1. Conversão para Pandas
        pdf = df.toPandas()

        # 2. Correção OBRIGATÓRIA de fuso horário (Evita erro fatal no PyArrow)
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)

        # 3. Conversão para Tabela Arrow
        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        # 4. Definição Dinâmica de Partições
        partition_by = None
        if particionar and camada == "bronze":
            possiveis = ["ano", "mes", "dia", "hora"]
            colunas_pdf = pdf.columns.tolist()
            partition_by = [c for c in possiveis if c in colunas_pdf]
            if not partition_by: 
                partition_by = None

        # 5. Normalização do dicionário para o SDK do delta-rs
        opts_sdk = {
            "account_name": storage_opts.get("account_name", "internshipdatalake"),
            "client_id": storage_opts.get("client_id"),
            "client_secret": storage_opts.get("client_secret"),
            "tenant_id": storage_opts.get("tenant_id")
        }

        # 6. Gravação física direta (Bypass do Databricks Serverless)
        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=modo_real,
            storage_options=opts_sdk,
            partition_by=partition_by,
            schema_mode="overwrite" if modo_real == "overwrite" else "merge"
        )
        
        print(f"[Sucesso] Gravado fisicamente via SDK em: {path} | Linhas: {len(pdf)}")
        return True

    except Exception as e:
        print(f"[Erro] Falha ao gravar {path}: {str(e)}")
        return False

In [0]:
# ==============================================================================
# CAMADA BRONZE
# ==============================================================================
import json
from datetime import datetime, timezone

# ==============================================================================
# FUNÇÕES AUXILIARES DE ADLS E CONTROLE (JSON) 
# ==============================================================================

def garantir_diretorio(container_client, caminho):
    """Cria diretório no ADLS se ainda não existir."""
    try:
        container_client.create_directory(caminho)
        print(f"Diretório criado: {caminho}")
    except Exception:
        pass # Diretório já existe

def ler_json_adls(container_client, caminho_arquivo, padrao=None):
    """Lê JSON no ADLS. Se não existir, retorna o padrão."""
    if padrao is None:
        padrao = []
    try:
        file_client = container_client.get_file_client(caminho_arquivo)
        conteudo = file_client.download_file().readall().decode("utf-8")
        if not conteudo.strip():
            return padrao
        return json.loads(conteudo)
    except Exception:
        return padrao

def salvar_json_adls(container_client, caminho_arquivo, dados):
    """Salva JSON no ADLS, criando diretórios pais se necessário."""
    pasta_pai = "/".join(caminho_arquivo.split("/")[:-1])
    garantir_diretorio(container_client, pasta_pai)

    file_client = container_client.get_file_client(caminho_arquivo)
    conteudo = json.dumps(dados, ensure_ascii=False, indent=2)
    file_client.upload_data(conteudo, overwrite=True)
    print(f"JSON de controle salvo em: {caminho_arquivo}")

def listar_controles_entidade(container_client, pasta_controle_base):
    """Lista todos os arquivos controle_leitura.json de uma entidade específica."""
    controles = []
    try:
        for item in container_client.get_paths(path=pasta_controle_base, recursive=True):
            if item.name.endswith("controle_leitura.json"):
                controles.append(item.name)
    except Exception:
        controles = []
    return controles

def carregar_arquivos_ja_lidos(container_client, pasta_controle_base):
    """Carrega arquivos já lidos considerando todos os controles existentes da entidade."""
    arquivos_lidos = set()
    controles = listar_controles_entidade(container_client, pasta_controle_base)

    for caminho_controle in controles:
        registros = ler_json_adls(container_client, caminho_controle, padrao=[])
        for r in registros:
            arquivo = r.get("arquivo_origem")
            status = r.get("status")
            if arquivo and status == "LIDO":
                arquivos_lidos.add(arquivo)

    return arquivos_lidos, controles

def extrair_data_do_caminho_raw(caminho):
    """Extrai data do padrão real-time-data/YYYY/MM/DD/HHMMSS/arquivo.parquet."""
    try:
        partes = caminho.split("/")
        ano, mes, dia = int(partes[1]), int(partes[2]), int(partes[3])
        hora_min_seg = partes[4]
        hora = int(hora_min_seg[0:2])
        minuto = int(hora_min_seg[2:4])
        segundo = int(hora_min_seg[4:6])
        return datetime(ano, mes, dia, hora, minuto, segundo, tzinfo=timezone.utc)
    except Exception:
        return datetime.min.replace(tzinfo=timezone.utc)

In [0]:
# ==============================================================================
# CAMADA SILVER
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

def ler_delta(camada, tabela, storage_opts):
    import pyspark.sql.functions as F
    
    # 1. Montagem dinâmica do caminho base
    if camada == "":
        caminho_base = f"squad1@internshipdatalake.dfs.core.windows.net/{tabela}"
    else:
        caminho_base = f"squad1@internshipdatalake.dfs.core.windows.net/{camada}/{tabela}"
        
    # Limpa barras duplicadas apenas no corpo do caminho, preservando o protocolo intacto
    caminho_limpo = caminho_base.replace("//", "/")
    caminho_final = f"abfss://{caminho_limpo}"
    
    # 2. Desempacota STORAGE_OPTIONS e mapeia as chaves OAuth do Hadoop para o Spark Reader
    account_name = storage_opts.get("account_name", "internshipdatalake")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")
    
    return (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .option("ignoreCorruptFiles", "true")
        .option("ignoreMissingFiles", "true")
        .load(caminho_final))
          
def schema_dq_logs():
    """
    Retorna o schema padronizado para a tabela de logs de qualidade de dados (DQ).
    """
    return StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])

In [0]:
# ==============================================================================
# DIAGNÓSTICO: explorar estrutura do squad3 e comparar schema com squad1
# ==============================================================================
from deltalake import DeltaTable
 
CONTAINER_SQUAD3 = "squad3"
 
def get_delta_path_squad3(camada: str, tabela: str, storage_opts: dict) -> str:
    """Mesma lógica do get_delta_path, mas apontando para o container squad3."""
    conta = storage_opts.get("account_name")
    corpo = tabela if not camada else f"{camada}/{tabela}"
    return f"abfss://{CONTAINER_SQUAD3}@{conta}.dfs.core.windows.net/{corpo}"
 
 
def listar_pastas_squad3(container_client_squad3, path: str = "", recursive: bool = True):
    """Lista arquivos/pastas dentro do container squad3."""
    paths = container_client_squad3.get_paths(path=path if path else None, recursive=recursive)
    for p in paths:
        tipo = "DIR " if p.is_directory else "FILE"
        print(f"{tipo} | {p.name}")
 

In [0]:
# ==============================================================================
# FEATURES DE ANOMALIA DE VENDAS (compartilhado entre treino e score)
# ==============================================================================
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def construir_features_pedidos(df_pedidos, df_itens, df_produtos):
    """
    Constrói a tabela de features em nível de PEDIDO (1 linha por id_pedido),
    usando o HISTÓRICO COMPLETO recebido (df_pedidos deve ser a Silver
    inteira, não um micro-lote) — as janelas de histórico do cliente
    (ticket médio, recência) precisam enxergar todos os pedidos anteriores
    daquele cliente, não só os que estão sendo pontuados agora.
    """
    df_itens_com_categoria = df_itens.join(
        df_produtos.select(
            F.col("sku"),
            F.col("id_categoria").alias("id_categoria_produto")
        ),
        "sku", "left"
    )

    df_agg_itens = df_itens_com_categoria.groupBy("id_pedido").agg(
        F.count("*").alias("qtd_itens"),
        F.countDistinct("sku").alias("qtd_skus_distintos"),
        F.countDistinct("id_categoria_produto").alias("qtd_categorias_distintas"),
        F.sum(F.coalesce(F.col("desconto_aplicado"), F.lit(0.0)) * F.col("quantidade")).alias("desconto_total")
    )

    w_cliente_historico = Window.partitionBy("id_cliente").orderBy("dt_pedido") \
        .rowsBetween(Window.unboundedPreceding, -1)
    w_cliente_ordem = Window.partitionBy("id_cliente").orderBy("dt_pedido")

    df_pedidos_hist = df_pedidos.withColumn(
        "ticket_medio_historico_cliente", F.avg("valor_total").over(w_cliente_historico)
    ).withColumn(
        "qtd_pedidos_anteriores_cliente", F.count("id_pedido").over(w_cliente_historico)
    ).withColumn(
        "dt_pedido_anterior", F.lag("dt_pedido").over(w_cliente_ordem)
    ).withColumn(
        "dias_desde_ultimo_pedido", F.datediff(F.col("dt_pedido"), F.col("dt_pedido_anterior"))
    )

    df_features = (
        df_pedidos_hist
        .join(df_agg_itens, "id_pedido", "left")
        .withColumn("hora_do_dia", F.hour("dt_pedido"))
        .withColumn("dia_da_semana", F.dayofweek("dt_pedido"))
        .withColumn("razao_frete_valor", F.col("valor_frete") / F.col("valor_total"))
        .withColumn(
            "desvio_pct_vs_media_cliente",
            F.when(
                F.col("ticket_medio_historico_cliente").isNotNull() & (F.col("ticket_medio_historico_cliente") > 0),
                (F.col("valor_total") - F.col("ticket_medio_historico_cliente")) / F.col("ticket_medio_historico_cliente")
            ).otherwise(F.lit(0.0))
        )
        .select(
            "id_pedido", "id_cliente",
            "valor_total", "valor_frete", "razao_frete_valor",
            "qtd_itens", "qtd_skus_distintos", "qtd_categorias_distintas", "desconto_total",
            "ticket_medio_historico_cliente", "qtd_pedidos_anteriores_cliente",
            "dias_desde_ultimo_pedido", "desvio_pct_vs_media_cliente",
            "hora_do_dia", "dia_da_semana",
            "metodo_pagamento"
        )
        .fillna({
            "ticket_medio_historico_cliente": 0.0,
            "qtd_pedidos_anteriores_cliente": 0,
            "dias_desde_ultimo_pedido": 0,
            "desconto_total": 0.0,
        })
    )
    return df_features


COLUNAS_NUMERICAS_ANOMALIA = [
    "valor_total", "valor_frete", "razao_frete_valor",
    "qtd_itens", "qtd_skus_distintos", "qtd_categorias_distintas", "desconto_total",
    "ticket_medio_historico_cliente", "qtd_pedidos_anteriores_cliente",
    "dias_desde_ultimo_pedido", "desvio_pct_vs_media_cliente",
    "hora_do_dia", "dia_da_semana",
]


# ==============================================================================
# PERSISTÊNCIA DE MODELOS ML NO ADLS (joblib via bytes)
# ==============================================================================
def salvar_modelo_ml(objeto, caminho_relativo: str, container_client):
    """Serializa um objeto Python (modelo, scaler, etc) via joblib e grava no ADLS."""
    import joblib
    import io
    buffer = io.BytesIO()
    joblib.dump(objeto, buffer)
    buffer.seek(0)
    file_client = container_client.get_file_client(caminho_relativo)
    file_client.upload_data(buffer.read(), overwrite=True)
    print(f"Modelo salvo em: {caminho_relativo}")


def carregar_modelo_ml(caminho_relativo: str, container_client):
    """Lê um objeto Python (modelo, scaler, etc) do ADLS, salvo via salvar_modelo_ml."""
    import joblib
    import io
    file_client = container_client.get_file_client(caminho_relativo)
    conteudo = file_client.download_file().readall()
    return joblib.load(io.BytesIO(conteudo))


# ==============================================================================
# AUTOENCODER (PyTorch) — import protegido, para não quebrar notebooks que
# não usam IA (Bronze/Silver/Gold de Data Quality não precisam de torch).
# ==============================================================================
try:
    import torch
    import torch.nn as nn
    _TORCH_DISPONIVEL = True
except ImportError:
    _TORCH_DISPONIVEL = False
 
 
if _TORCH_DISPONIVEL:
    class AutoencoderTorch(nn.Module):
        """
        Mesma arquitetura conceitual das Etapas 2-4 (encoder -> gargalo ->
        decoder, ReLU nas camadas internas, saída linear).
        """
        def __init__(self, dim_entrada: int, dim_oculta1: int, dim_gargalo: int):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(dim_entrada, dim_oculta1),
                nn.ReLU(),
                nn.Linear(dim_oculta1, dim_gargalo),
                nn.ReLU(),
            )
            self.decoder = nn.Sequential(
                nn.Linear(dim_gargalo, dim_oculta1),
                nn.ReLU(),
                nn.Linear(dim_oculta1, dim_entrada),
            )
 
        def forward(self, x):
            z = self.encoder(x)
            return self.decoder(z)
else:
    class AutoencoderTorch:
        """
        Placeholder: só lança erro (com mensagem clara) se alguém tentar usar
        sem o torch instalado — em vez de quebrar o utils.py inteiro na
        importação, para quem só está usando Bronze/Silver/Gold.
        """
        def __init__(self, *args, **kwargs):
            raise ImportError(
                "PyTorch não está instalado nesta sessão. Rode "
                "'%pip install torch' e reinicie o Python antes de usar "
                "AutoencoderTorch/salvar_modelo_pytorch/carregar_modelo_pytorch."
            )
 
 
def salvar_modelo_pytorch(modelo, caminho_relativo: str, container_client):
    """Salva os PESOS do modelo PyTorch (state_dict) no ADLS."""
    import torch  # import local: só exige torch quando esta função é chamada
    import io
    buffer = io.BytesIO()
    torch.save(modelo.state_dict(), buffer)
    buffer.seek(0)
    file_client = container_client.get_file_client(caminho_relativo)
    file_client.upload_data(buffer.read(), overwrite=True)
    print(f"Modelo PyTorch salvo em: {caminho_relativo}")
 
 
def carregar_modelo_pytorch(caminho_relativo: str, dim_entrada: int, dim_oculta1: int,
                             dim_gargalo: int, container_client):
    """Recria a arquitetura e carrega os pesos salvos."""
    import torch  # import local: só exige torch quando esta função é chamada
    import io
    file_client = container_client.get_file_client(caminho_relativo)
    conteudo = file_client.download_file().readall()
 
    modelo = AutoencoderTorch(dim_entrada, dim_oculta1, dim_gargalo)
    modelo.load_state_dict(torch.load(io.BytesIO(conteudo), map_location="cpu"))
    modelo.eval()
    return modelo